# Inférence statistique

## Auto-ML

pycaret ne supporte pas encore officiellement Python 3.14.3 donc on utilise à la place le package flaml

In [49]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from flaml import AutoML

In [50]:
df_train = pd.read_csv("../data_finale/featuring/train_featured.csv")
df_val = pd.read_csv("../data_finale/featuring/val_featured.csv")
df_test = pd.read_csv("../data_finale/featuring/test_featured.csv")

In [51]:
colonne_cible = "market_value_in_eur"
colonne_joueur = "player"
colonne_team = "team"
colonne_nation = "nation"

df_train = df_train.dropna(subset=[colonne_cible])
df_val = df_val.dropna(subset=[colonne_cible])
df_test = df_test.dropna(subset=[colonne_cible])

# Séparation des features et de la variable cible
joueurs_test = df_test[colonne_joueur]

# On supprime les colonnes texte et la cible pour l'entraînement
X_train = df_train.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_train = df_train[colonne_cible]

X_val = df_val.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_val = df_val[colonne_cible]

X_test = df_test.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_test = df_test[colonne_cible]

In [ ]:
print("--- Entraînement FLAML ---")
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    task="regression",
    metric="mae",
    time_budget=300,  # 5 minutes pour FLAML
    estimator_list=["xgboost", "lgbm", "catboost", "rf", "extra_tree", "kneighbor", "enet", "lassolars"],
    seed=1308
)

# Récupération des meilleures prédictions de FLAML
preds_flaml = automl.predict(X_test)
mae_flaml = mean_absolute_error(y_test, preds_flaml)


# Préparation : Sélection dynamique des colonnes sans aucun NaN

# On ne garde que les colonnes où la somme des NaN est égale à 0
colonnes_sans_nan = X_train.columns[X_train.isna().sum() == 0].tolist()

print(f"Filtrage pour les modèles linéaires/SVR :")
print(f" -> {len(colonnes_sans_nan)} colonnes conservées sur {X_train.shape[1]} (0 NaN).")

# Création des sous-ensembles spécifiques "propres"
X_train_sans_nan = X_train[colonnes_sans_nan]
X_test_sans_nan = X_test[colonnes_sans_nan]


print("\n--- Entraînement SVR ---")

# Plus besoin d'imputeur ! On garde juste le StandardScaler (vital pour le SVR)
svr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

# Ajustement de la grille (préfixe svr__ nécessaire à cause du Pipeline)
param_svr = {
    'svr__C': [0.1, 1, 10, 100],
    'svr__kernel': ['rbf', 'linear'],
    'svr__epsilon': [0.01, 0.1, 0.5]
}

grid_svr = GridSearchCV(svr_pipeline, param_svr, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
# On entraîne sur les colonnes sans NaN
grid_svr.fit(X_train_sans_nan, y_train)

preds_svr = grid_svr.predict(X_test_sans_nan)
mae_svr = mean_absolute_error(y_test, preds_svr)


print("--- Entraînement Régression Linéaire ---")

# Optionnel mais recommandé : Mettre un StandardScaler ici aussi si vous utilisez 
# des variables aux échelles très différentes, ou laisser LinearRegression brute.
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# On entraîne également sur les colonnes sans NaN
lr_pipeline.fit(X_train_sans_nan, y_train)

preds_lr = lr_pipeline.predict(X_test_sans_nan)
mae_lr = mean_absolute_error(y_test, preds_lr)



print("\nComparaison des performences (MAE)")
print(f"FLAML : {mae_flaml:,.0f} € (Modèle : {automl.best_estimator})")
print(f"SVR (Optimisé par GridSearch)        : {mae_svr:,.0f} €")
print(f"Régression Linéaire Multiple         : {mae_lr:,.0f} €")

# Trouver le grand gagnant
resultats = {
    "FLAML": (mae_flaml, preds_flaml),
    "SVR": (mae_svr, preds_svr),
    "LinearRegression": (mae_lr, preds_lr)
}
meilleur_approche = min(resultats, key=lambda k: resultats[k][0])
print(f"\nLe meilleur choix final est : {meilleur_approche}")

--- Entraînement FLAML ---
[flaml.automl.logger: 06-30 15:04:58] {2375} INFO - task = regression
[flaml.automl.logger: 06-30 15:04:58] {2383} INFO - Data split method: uniform
[flaml.automl.logger: 06-30 15:04:58] {2386} INFO - Evaluation method: holdout
[flaml.automl.logger: 06-30 15:04:58] {2489} INFO - Minimizing error metric: mae
[flaml.automl.logger: 06-30 15:04:58] {2606} INFO - List of ML learners in AutoML Run: ['xgboost', 'lgbm', 'catboost', 'rf', 'extra_tree', 'kneighbor', 'enet', 'lassolars']
[flaml.automl.logger: 06-30 15:04:58] {2911} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 06-30 15:04:58] {3046} INFO - Estimated sufficient time budget=918s. Estimated necessary time budget=3s.
[flaml.automl.logger: 06-30 15:04:58] {3097} INFO -  at 0.4s,	estimator xgboost's best error=9.5835e+06,	best estimator xgboost's best error=9.5835e+06
[flaml.automl.logger: 06-30 15:04:58] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 06-30 15:04:58]

In [ ]:
# On extrait le vrai modèle CatBoost entraîné par FLAML
meilleur_catboost = automl.model.estimator

# On affiche tous ses hyperparamètres sous forme de dictionnaire
print("\nHyperparamètres finaux")
print(meilleur_catboost.get_all_params())


=== HYPERPARAMÈTRES EXACTS DU CATBOOST ===
{'nan_mode': 'Min', 'eval_metric': 'RMSE', 'iterations': 8192, 'sampling_frequency': 'PerTree', 'leaf_estimation_method': 'Newton', 'od_pval': 0, 'random_score_type': 'NormalWithModelSizeDecrease', 'grow_policy': 'SymmetricTree', 'penalties_coefficient': 1, 'boosting_type': 'Plain', 'model_shrink_mode': 'Constant', 'feature_border_type': 'GreedyLogSum', 'bayesian_matrix_reg': 0.10000000149011612, 'eval_fraction': 0, 'force_unit_auto_pair_weights': False, 'l2_leaf_reg': 3, 'random_strength': 1, 'od_type': 'Iter', 'rsm': 1, 'boost_from_average': True, 'model_size_reg': 0.5, 'pool_metainfo_options': {'tags': {}}, 'subsample': 0.800000011920929, 'use_best_model': True, 'od_wait': 42, 'random_seed': 10242048, 'depth': 6, 'posterior_sampling': False, 'border_count': 254, 'classes_count': 0, 'auto_class_weights': 'None', 'sparse_features_conflict_fraction': 0, 'leaf_estimation_backtracking': 'AnyImprovement', 'best_model_min_trees': 1, 'model_shrink

## Premiers tests de modélisation

In [52]:
modeles = {
    
    # Forêts (On garde une bonne profondeur)
    "Random Forest": RandomForestRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=200,
        max_depth=30,
        min_samples_split=5
    ),
    "Extra Trees": ExtraTreesRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=200,
        max_depth=30,
        min_samples_split=5
    ),
    
    # Boosting (Alignés sur la stratégie gagnante de FLAML)
    "XGBoost": XGBRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=2000,       # Augmenté pour compenser le learning rate plus bas
        learning_rate=0.01,       # Plus robuste
        max_depth=6
    ),
    "LightGBM": LGBMRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=3000,       # Les arbres de LightGBM sont très rapides à construire
        learning_rate=0.01,
        max_depth=6,
        verbose=-1
    ),
    "CatBoost": CatBoostRegressor(
        random_state=1308,
        iterations=8192,
        learning_rate=0.005,
        depth=6,
        l2_leaf_reg=3,
        subsample=0.8,
        early_stopping_rounds=100,
        verbose=0                 
    ),
}

In [53]:
# Entraînement puis évaluation
resultats = {}

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_val = modele.predict(X_val)
    
    mae_val = mean_absolute_error(y_val, preds_val)
    r2_val = r2_score(y_val, preds_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, preds_val))
    mape_val = mean_absolute_percentage_error(y_val, preds_val)

    # R² Ajusté
    n = len(y_val)          # Nombre d'observations
    p = X_val.shape[1]      # Nombre de variables (colonnes)
    r2_ajuste_val = 1 - (1 - r2_val) * (n - 1) / (n - p - 1)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Val": mae_val,
        "RMSE Val": rmse_val,
        "MAPE Val": mape_val,
        "R² Val": r2_val,
        "R² Ajusté Val": r2_ajuste_val
    }
    
    print(f"   -> Validation | MAE : {mae_val:,.0f} € | RMSE : {rmse_val:,.0f} € | MAPE : {mape_val:.2%} | R² : {r2_val:.2%}")

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Val (MAE)": f"{metrics['MAE Val']} €",
        "Score R² Val": f"{metrics['R² Val']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Val (MAE)", ascending=True)
print(df_final.to_string(index=False))

Entraînement de Random Forest...
   -> Validation | MAE : 5,040,843 € | RMSE : 9,713,330 € | MAPE : 89.33% | R² : 70.90%
Entraînement de Extra Trees...
   -> Validation | MAE : 4,983,436 € | RMSE : 9,540,966 € | MAPE : 91.67% | R² : 71.92%
Entraînement de XGBoost...
   -> Validation | MAE : 4,749,971 € | RMSE : 8,984,303 € | MAPE : 87.71% | R² : 75.10%
Entraînement de LightGBM...
   -> Validation | MAE : 4,782,923 € | RMSE : 9,010,973 € | MAPE : 88.97% | R² : 74.95%
Entraînement de CatBoost...
   -> Validation | MAE : 4,659,503 € | RMSE : 8,835,834 € | MAPE : 91.54% | R² : 75.92%
CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Val (MAE) Score R² Val
     CatBoost      4659502.658309187 €       75.92%
      XGBoost      4749970.617379152 €       75.10%
     LightGBM     4782922.8956359895 €       74.95%
  Extra Trees      4983435.938063114 €       71.92%
Random Forest      5040843.400330284 €       70.90%


In [54]:
# Entraînement puis évaluation
resultats = {}

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_test = modele.predict(X_test)
    
    mae_test = mean_absolute_error(y_test, preds_test)
    r2_test = r2_score(y_test, preds_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, preds_test))
    mape_test = mean_absolute_percentage_error(y_test, preds_test)

    # R² Ajusté
    n = len(y_test)          # Nombre d'observations
    p = X_test.shape[1]      # Nombre de variables (colonnes)
    r2_ajuste_test = 1 - (1 - r2_test) * (n - 1) / (n - p - 1)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Test": mae_test,
        "R² Test": r2_test,
        "MAPE Test": mape_test,
        "R² Test": r2_test,
        "R² Ajusté Test": r2_ajuste_test
    }
    
    print(f"   -> Test | MAE : {mae_test:,.0f} € | RMSE : {rmse_test:,.0f} € | MAPE : {mape_test:.2%} | R² : {r2_test:.2%}")

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Test (MAE)": f"{metrics['MAE Test']:,.0f} €",
        "Score R² Test": f"{metrics['R² Test']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Test (MAE)", ascending=True)
print(df_final.to_string(index=False))

Entraînement de Random Forest...
   -> Test | MAE : 5,378,551 € | RMSE : 10,630,318 € | MAPE : 80.18% | R² : 67.69%
Entraînement de Extra Trees...
   -> Test | MAE : 5,331,168 € | RMSE : 10,490,978 € | MAPE : 83.06% | R² : 68.53%
Entraînement de XGBoost...
   -> Test | MAE : 5,113,909 € | RMSE : 9,909,866 € | MAPE : 78.50% | R² : 71.92%
Entraînement de LightGBM...
   -> Test | MAE : 5,073,985 € | RMSE : 9,701,387 € | MAPE : 79.53% | R² : 73.09%
Entraînement de CatBoost...
   -> Test | MAE : 5,062,656 € | RMSE : 9,765,905 € | MAPE : 78.63% | R² : 72.73%
CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Test (MAE) Score R² Test
     CatBoost               5,062,656 €        72.73%
     LightGBM               5,073,985 €        73.09%
      XGBoost               5,113,909 €        71.92%
  Extra Trees               5,331,168 €        68.53%
Random Forest               5,378,551 €        67.69%
